# Retail Sales Analytics
Portfolio project: data quality, seasonal sales, product performance, and geographic sales. Data: UCI Online Retail (2010–2011). Gross sales exclude returns; not net revenue.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import requests, zipfile, io, sqlite3
url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
response = requests.get(url, timeout=120)
response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    filename = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
    with z.open(filename) as f:
        df = pd.read_excel(f)
print(df.shape)
display(df.head())

## Data quality audit
Inspect issues before filtering. Exact duplicates may be legitimate separate line items.

In [ ]:
print("Missing values:\n", df.isna().sum())
print("Duplicates:", df.duplicated().sum())
print("Negative quantities:", (df.Quantity < 0).sum())
print("Nonpositive prices:", (df.UnitPrice <= 0).sum())
print("Cancellation rows:", df.InvoiceNo.astype(str).str.startswith("C").sum())

## Define positive-sale and merchandise datasets
Retain the original dataframe. This analysis does not calculate net revenue.

In [ ]:
df_clean = df.loc[(df.Quantity > 0) & (df.UnitPrice > 0) & ~df.InvoiceNo.astype(str).str.startswith("C")].copy()
df_clean["Description"] = df_clean.Description.fillna("Unknown Product")
df_clean["SalesAmount"] = df_clean.Quantity * df_clean.UnitPrice
df_clean["YearMonth"] = pd.to_datetime(df_clean.InvoiceDate).dt.to_period("M")
non_merchandise_codes = {"DOT", "POST", "M", "D", "C2", "BANK CHARGES"}
df_products = df_clean.loc[~df_clean.StockCode.astype(str).isin(non_merchandise_codes)].copy()
print("Filtered rows:",len(df_clean),"Merchandise rows:",len(df_products))

## Monthly sales trend

In [ ]:
monthly_sales = df_clean.groupby("YearMonth").SalesAmount.sum().sort_index()
ax = monthly_sales.plot(marker="o", figsize=(11,5), title="Monthly gross sales (positive sales, before returns)")
ax.set_ylabel("Gross sales (£)")
ax.set_xlabel("Month")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Order volume vs average order value

In [ ]:
subset = df_products.loc[df_products.YearMonth.astype(str).isin(["2011-08","2011-11"])]
kpis = subset.groupby("YearMonth").agg(GrossSales=("SalesAmount","sum"), TotalOrders=("InvoiceNo","nunique"), UnitsSold=("Quantity","sum"))
kpis["AvgOrderValue"] = kpis.GrossSales/kpis.TotalOrders
display(kpis.round(2))
print("Growth (%):")
display(((kpis.iloc[1]/kpis.iloc[0]-1)*100).round(2))

## Christmas-labeled product contribution

In [ ]:
seasonal = subset.copy()
seasonal["IsChristmas"] = seasonal.Description.str.contains("CHRISTMAS", case=False, na=False)
t = seasonal.groupby(["YearMonth","IsChristmas"]).SalesAmount.sum().unstack(fill_value=0)
display(t.round(2))
print("Contribution to gross-sales growth (%):",round((t.iloc[1].get(True,0)-t.iloc[0].get(True,0))/(t.iloc[1].sum()-t.iloc[0].sum())*100,2))

## Product ranking and outlier sensitivity

In [ ]:
top = df_products.groupby("Description").SalesAmount.sum().nlargest(10)
display(top.round(2))
top.sort_values().plot(kind="barh",figsize=(10,6),title="Top merchandise descriptions by gross sales")
plt.tight_layout();plt.show()
invoice = df.loc[df.InvoiceNo.astype(str)=="581483"]
display(invoice[["InvoiceNo","StockCode","Description","Quantity","UnitPrice","CustomerID"]])
print("Potential negative rows for stock 23843:")
display(df.loc[(df.StockCode.astype(str)=="23843") & (df.Quantity<0), ["InvoiceNo","Quantity","InvoiceDate"]].head(20))
without = df_products.loc[df_products.InvoiceNo.astype(str)!="581483"]
print("Impact of excluding invoice (%):",round((1-without.SalesAmount.sum()/df_products.SalesAmount.sum())*100,2))

## SQL analysis in SQLite

In [ ]:
conn=sqlite3.connect(":memory:")
sql_df=df_products.copy()
sql_df["InvoiceDate"]=sql_df.InvoiceDate.astype(str)
sql_df["YearMonth"]=sql_df.YearMonth.astype(str)
sql_df.to_sql("retail_sales",conn,if_exists="replace",index=False)
query="""SELECT Country, ROUND(SUM(SalesAmount),2) AS TotalSales, COUNT(DISTINCT InvoiceNo) AS TotalOrders FROM retail_sales GROUP BY Country ORDER BY TotalSales DESC LIMIT 10"""
display(pd.read_sql_query(query,conn))

## Conclusions and caveats
- Order count, rather than average order value, drove August–November gross-sales growth.
- Christmas-labeled merchandise contributed a measurable portion of the increase; causation is unproven.
- Product rankings are sensitive to an unusually large invoice.
- Historical data, incomplete December 2011, unreconciled returns, possible duplicate line items, and heuristic merchandise classification limit interpretation.